# Training Hugging Face Models with RLT

This notebook demonstrates how to train a Hugging Face model using the RLT (Reinforcement Learning Teachers) methodology.

## Key Components:
1. **Trainable Teacher Model**: Any HF model (Llama, Phi, GPT, etc.)
2. **GRPO Training**: Group Relative Policy Optimization
3. **Dense Rewards**: Student understanding (rSS) - KL divergence (rKL)
4. **LoRA/PEFT**: Efficient training with parameter-efficient fine-tuning

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers torch datasets accelerate peft
!pip install -q bitsandbytes trl matplotlib tqdm

import os
import sys
sys.path.append(os.path.abspath('.'))

print("✅ Dependencies installed")

## 2. Quick Training Example

In [ ]:
# Quick training with default settings
!python train_rlt_model.py \
    --teacher-model "microsoft/phi-2" \
    --datasets gsm8k \
    --max-samples 100 \
    --num-epochs 1 \
    --batch-size 2 \
    --use-lora

## 3. Custom Training Pipeline

In [ ]:
from src.models.hf_teacher_model import HFTeacherModel
from src.training.grpo_trainer import GRPOTrainer, GRPOConfig
from src.rewards import RLTRewardFunction, LocalStudentEvaluator, TransformerKLCalculator
from src.data import DataLoader as RLTDataLoader

# 1. Initialize teacher model (any HF model)
teacher = HFTeacherModel(
    model_name="microsoft/phi-2",  # Can use: meta-llama/Llama-3.2-3B, etc.
    use_lora=True,
    lora_config={
        "r": 32,
        "lora_alpha": 64,
        "lora_dropout": 0.1
    }
)

print("✅ Teacher model loaded with LoRA")
teacher.model.print_trainable_parameters()

In [ ]:
# 2. Setup reward components
student = LocalStudentEvaluator(model_name="microsoft/phi-2")
kl_calc = TransformerKLCalculator(model_name="gpt2")
reward_fn = RLTRewardFunction(student, kl_calc)

print("✅ Reward system initialized")

In [ ]:
# 3. Load data
data_loader = RLTDataLoader()
train_data = data_loader.load_dataset('gsm8k', split='train', max_samples=50)

print(f"✅ Loaded {len(train_data)} training samples")
print(f"\nExample:")
print(f"Q: {train_data[0].question[:100]}...")
print(f"A: {train_data[0].answer}")

In [ ]:
# 4. Test generation before training
print("Testing generation before training...\n")

result = teacher.generate_explanation(
    question=train_data[0].question,
    answer=train_data[0].answer,
    temperature=0.7
)

print(f"Generated explanation:\n{result['explanation'][:200]}...")

# Compute initial reward
reward_result = reward_fn.compute_reward(
    [result['explanation']],
    [train_data[0].question]
)

print(f"\nInitial reward: {reward_result['rewards'][0]:.4f}")

## 4. Configure and Run Training

In [ ]:
from torch.utils.data import DataLoader
from train_rlt_model import RLTDataset, collate_fn

# Create dataset and dataloader
dataset = RLTDataset(train_data, None)
dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

# Configure training
config = GRPOConfig(
    learning_rate=2e-5,
    batch_size=2,
    num_epochs=1,
    group_size=3,  # Generate 3 explanations per question
    gradient_accumulation_steps=4,
    save_steps=10,
    checkpoint_dir="./demo_checkpoints"
)

# Initialize trainer
trainer = GRPOTrainer(
    teacher_model=teacher,
    student_evaluator=student,
    reward_function=reward_fn,
    config=config
)

print("✅ Training configured")
print(f"Total batches: {len(dataloader)}")
print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")

In [ ]:
# Run training
print("Starting training...\n")

trainer.train(
    train_dataloader=dataloader,
    num_epochs=1
)

print("\n✅ Training completed!")

In [ ]:
# Test generation after training
print("Testing generation after training...\n")

result_after = teacher.generate_explanation(
    question=train_data[0].question,
    answer=train_data[0].answer,
    temperature=0.7
)

print(f"Generated explanation:\n{result_after['explanation'][:200]}...")

# Compute reward after training
reward_after = reward_fn.compute_reward(
    [result_after['explanation']],
    [train_data[0].question]
)

print(f"\nReward after training: {reward_after['rewards'][0]:.4f}")
print(f"Improvement: {reward_after['rewards'][0] - reward_result['rewards'][0]:.4f}")

## 5. Save Trained Model

In [ ]:
# Save the trained model
output_dir = "./trained_rlt_model"
teacher.save_model(output_dir)

print(f"✅ Model saved to: {output_dir}")
print("\nTo load the model later:")
print(f'model = AutoModelForCausalLM.from_pretrained("{output_dir}")')

## 6. Advanced Training Options

### Using Different Models

In [ ]:
# Examples of different models you can use
model_options = [
    "microsoft/phi-2",                    # 2.7B params
    "microsoft/phi-1_5",                  # 1.3B params
    "meta-llama/Llama-3.2-3B-Instruct",  # 3B params (requires access)
    "meta-llama/Llama-3.2-1B-Instruct",  # 1B params (requires access)
    "google/gemma-2b",                    # 2B params
    "EleutherAI/pythia-1.4b",            # 1.4B params
    "bigscience/bloom-1b7",               # 1.7B params
]

print("Available models for RLT training:")
for model in model_options:
    print(f"  - {model}")

### Custom LoRA Configuration

In [ ]:
# Advanced LoRA configuration
advanced_lora_config = {
    "r": 64,              # Rank (higher = more capacity)
    "lora_alpha": 128,    # Scaling factor
    "lora_dropout": 0.05, # Dropout for regularization
    "bias": "none",       # Don't train biases
    "task_type": "CAUSAL_LM"
}

# Target specific modules based on model architecture
# For Llama models: ["q_proj", "v_proj", "k_proj", "o_proj"]
# For GPT models: ["c_attn", "c_proj"]
# For Phi models: ["q_proj", "v_proj", "k_proj", "dense"]

## 7. Training Tips

### Memory Optimization
- Use 8-bit quantization: `use_8bit=True`
- Reduce batch size: `--batch-size 1`
- Increase gradient accumulation: `gradient_accumulation_steps=8`
- Use smaller models for student evaluation

### Performance Tips
- Start with smaller datasets (100-500 samples)
- Use lower group sizes (2-3) for faster training
- Monitor rewards to ensure convergence
- Save checkpoints frequently

### Quality Tips
- Use diverse temperature sampling (0.6-0.9)
- Balance rSS and rKL with appropriate lambda
- Evaluate on held-out test sets
- Compare with baseline models

## Summary

You now have a complete system for training any Hugging Face model using RLT:

1. **Model Loading**: Load any HF model with LoRA
2. **GRPO Training**: Optimize for student understanding
3. **Dense Rewards**: Balance solution score and KL divergence
4. **Efficient Training**: Use LoRA and 8-bit quantization
5. **Flexible Architecture**: Support for various model types

The trained models will generate better teaching explanations that maximize student understanding!